# PINNsFormer + Chebyshev-4: Navier-Stokes x3 Seeds

Self-contained notebook. No external `chebyshev_runner.py`.
Defaults: `N_TRAIN=600`, seeds 0/1/2, 1000 steps, Adam lr=1e-4, 4 losses.


In [ ]:
from pathlib import Path
import csv,gc,json,random,sys,time
import numpy as np, torch, matplotlib.pyplot as plt
from tqdm.auto import tqdm

ROOT=Path('/home/simplexity/cyt/pinnsformer-main')
EXP=ROOT/'pinnsformer-config'
DATA=ROOT/'demo'/'navier_stokes'/'cylinder_nektar_wake.mat'
OUT=EXP/'outputs'/'navier_stokes_chebyshev4'
AGG=OUT/'aggregate'
OUT.mkdir(parents=True,exist_ok=True); AGG.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(EXP))
from navier_stokes_common import PINNsformer,init_weights,get_n_params,load_training_data,compute_ns_losses,evaluation_tensors
from gradient_diagnostics import gradient_vector,pairwise_cosine_matrix

P=4; SEEDS=[0,1,2]; N_TRAIN=600; EPOCHS=1000; LR=1e-4
NUM_STEP=5; TIME_STEP=1e-2; SNAP=100; METRIC_INTERVAL=10
LOSS_KEYS=['u_data','v_data','f_u','f_v']
FW_MAX_ITERS=100; FW_TOL=1e-6; STOP_TOL=1e-6; EPS=1e-12
DEVICE='cuda:0' if torch.cuda.is_available() else 'cpu'
print('P=',P,'N_TRAIN=',N_TRAIN,'DEVICE=',DEVICE,'OUT=',OUT)


In [ ]:
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()

def model_new():
    m=PINNsformer(d_out=2,d_hidden=512,d_model=32,N=1,heads=2).to(DEVICE); m.apply(init_weights); return m

def rows_write(path,rows):
    if not rows:return
    keys=[]
    for r in rows:
        for k in r:
            if k not in keys: keys.append(k)
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=keys); w.writeheader(); w.writerows(rows)

def rows_read(path):
    with Path(path).open(encoding='utf-8') as f:return list(csv.DictReader(f))

def mean(x):
    a=np.asarray(x,float); a=a[np.isfinite(a)]; return float(a.mean()) if len(a) else float('nan')

def std(x):
    a=np.asarray(x,float); a=a[np.isfinite(a)]
    return float(a.std(ddof=1)) if len(a)>1 else (0.0 if len(a)==1 else float('nan'))

@torch.no_grad()
def apply_grad(m,g):
    o=0
    for p in m.parameters():
        if not p.requires_grad:continue
        n=p.numel(); p.grad=g[o:o+n].view_as(p).clone(); o+=n

def lp(x,p): return torch.linalg.vector_norm(x,ord=p)

def primal(w,p):
    n=lp(w,p)
    if float(n)<=EPS:return torch.zeros_like(w),n
    return (w/n if p==2 else torch.sign(w)*torch.abs(w).pow(3)/n.pow(3)),n

def line_search(w,z,p):
    d=z-w
    if p==2:
        den=torch.dot(d,d)
        return 0.0 if float(den)<=EPS else float((-torch.dot(w,d)/den).clamp(0,1))
    c3=float(torch.sum(d**4)); c2=float(3*torch.sum(w*d**3)); c1=float(3*torch.sum(w**2*d**2)); c0=float(torch.sum(w**3*d))
    h=lambda g:((c3*g+c2)*g+c1)*g+c0
    if h(0)>=0:return 0.0
    if h(1)<=0:return 1.0
    lo,hi=0.,1.
    for _ in range(40):
        m=(lo+hi)/2
        if h(m)<=0:lo=m
        else:hi=m
    return (lo+hi)/2

def dual(ghat,p,a0=None):
    m=ghat.shape[0]
    a=torch.full((m,),1/m,device=ghat.device,dtype=ghat.dtype) if a0 is None else a0.detach().clone()
    a=a.clamp_min(0); a=a/a.sum().clamp_min(EPS)
    gap=float('inf'); it=0
    for k in range(FW_MAX_ITERS):
        it=k+1; w=(a[:,None]*ghat).sum(0); v,n=primal(w,p)
        if float(n)<=EPS: gap=0.; break
        ga=ghat@v; j=int(torch.argmin(ga)); gap=float(torch.dot(a,ga)-ga[j])
        if gap<=FW_TOL:break
        g=line_search(w,ghat[j],p)
        if g<=EPS:break
        a.mul_(1-g); a[j]+=g
    w=(a[:,None]*ghat).sum(0); _,n=primal(w,p)
    return a,w,it,gap,float(n)

def cheb(gs,p,a0=None):
    norms=torch.linalg.vector_norm(gs,ord=p,dim=1); gh=gs/norms.clamp_min(EPS)[:,None]
    a,w,it,gap,dn=dual(gh,p,a0); v,n=primal(w,p)
    if float(n)<=STOP_TOL:return None,a,{'fw_iters':it,'fw_gap':gap,'dual_norm':dn,'min_raw_alignment':0.,'min_normalized_alignment':0.}
    ra=gs@v; na=gh@v; s=ra.sum()
    return s*v,a,{'fw_iters':it,'fw_gap':gap,'dual_norm':dn,'min_raw_alignment':float(ra.min()),'min_normalized_alignment':float(na.min())}

def four_grads(m,b):
    L=compute_ns_losses(m,b['x'],b['y'],b['t'],b['u'],b['v']); G={}
    for i,k in enumerate(LOSS_KEYS):G[k]=gradient_vector(L[k],m,retain_graph=i<3).detach()
    return L,G

def geom(G):
    names,M=pairwise_cosine_matrix(G); q=[float(M[i,j]) for i in range(4) for j in range(i+1,4) if np.isfinite(M[i,j])]
    return mean(q),min(q),float(np.mean(np.array(q)<0))

def rel(pred,true):
    pred=np.asarray(pred).ravel(); true=np.asarray(true).ravel()
    return float(np.linalg.norm(pred-true)/np.linalg.norm(true))

def evaluate(m,ref):
    t,truth=evaluation_tensors(ref,DEVICE,snap=SNAP,num_step=NUM_STEP,time_step=TIME_STEP)
    out=m(t['x'],t['y'],t['t']); psi=out[:,:,0:1]; pp=out[:,0,1].detach().cpu().numpy()
    up=torch.autograd.grad(psi,t['y'],torch.ones_like(psi),retain_graph=True)[0][:,0].detach().cpu().numpy()
    vp=-torch.autograd.grad(psi,t['x'],torch.ones_like(psi))[0][:,0].detach().cpu().numpy()
    ut=np.asarray(truth['u']).ravel(); vt=np.asarray(truth['v']).ravel(); pt=np.asarray(truth['p']).ravel(); pp=pp.ravel(); pp-=np.mean(pp-pt)
    return {'rel_l2_u':rel(up,ut),'rel_l2_v':rel(vp,vt),'rel_l2_p_aligned':rel(pp,pt)}


In [ ]:
def run_seed(seed):
    seed_all(seed)
    if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    b,ref=load_training_data(DATA,DEVICE,seed=seed,n_train=N_TRAIN,num_step=NUM_STEP,time_step=TIME_STEP)
    m=model_new(); opt=torch.optim.Adam(m.parameters(),lr=LR); hist=[]; a0=None; solver_s=0.
    sync(); t0=time.perf_counter()
    for step in tqdm(range(EPOCHS),desc=f'Cheb-{P} seed={seed}'):
        opt.zero_grad(set_to_none=True); L,G=four_grads(m,b); gs=torch.stack([G[k] for k in LOSS_KEYS])
        ts=time.perf_counter(); d,a,info=cheb(gs,P,a0); sync(); solver_s+=time.perf_counter()-ts; a0=a.detach()
        if d is None: break
        apply_grad(m,d); opt.step()
        if step%METRIC_INTERVAL==0 or step==EPOCHS-1:
            cmn,cmin,cr=geom(G)
            r={'seed':seed,'step':step,'total':float(L['total']),'u_data':float(L['u_data']),'v_data':float(L['v_data']),'f_u':float(L['f_u']),'f_v':float(L['f_v']),'physics':float(L['physics']),
               'raw_cosine_mean':cmn,'raw_cosine_min':cmin,'raw_conflict_rate':cr,**info}
            for i,k in enumerate(LOSS_KEYS):r['alpha_'+k]=float(a[i]); r['grad_l2_'+k]=float(G[k].norm())
            hist.append(r)
    sync(); wall=time.perf_counter()-t0; ev=evaluate(m,ref); F=compute_ns_losses(m,b['x'],b['y'],b['t'],b['u'],b['v'])
    met={'method':f'chebyshev{P}','p':P,'seed':seed,'n_train':N_TRAIN,'wall_seconds':wall,'solver_seconds':solver_s,
         'peak_vram_mb':float(torch.cuda.max_memory_allocated()/1048576) if torch.cuda.is_available() else 0.,
         'n_params':get_n_params(m),'final_total_loss':float(F['total']),'final_physics_loss':float(F['physics']),**ev}
    rd=OUT/f'seed_{seed}'; rd.mkdir(exist_ok=True); rows_write(rd/'history.csv',hist); rows_write(rd/'metrics.csv',[met]); torch.save(m.state_dict(),rd/'model.pt')
    del m,opt,b; gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return met

metrics=[run_seed(s) for s in SEEDS]
rows_write(AGG/'all_field_metrics.csv',metrics)
hist=[]
for s in SEEDS: hist+=rows_read(OUT/f'seed_{s}'/'history.csv')
rows_write(AGG/'all_metric_history.csv',hist)

summary=[]
for k in ['rel_l2_u','rel_l2_v','rel_l2_p_aligned','final_total_loss','final_physics_loss','wall_seconds','solver_seconds','peak_vram_mb']:
    v=[float(r[k]) for r in metrics]; summary.append({'method':f'Chebyshev-{P}','p':P,'n_train':N_TRAIN,'metric':k,'mean':mean(v),'std':std(v)})
rows_write(AGG/'mean_std_summary.csv',summary)
for r in summary: print(r)


In [ ]:
for s in SEEDS:
    rs=sorted([r for r in hist if int(r['seed'])==s],key=lambda r:int(r['step'])); x=[int(r['step']) for r in rs]
    plt.figure(figsize=(7,4))
    for k in LOSS_KEYS: plt.semilogy(x,[float(r[k]) for r in rs],label=k)
    plt.semilogy(x,[float(r['physics']) for r in rs],label='physics',lw=2); plt.legend(ncol=2); plt.xlabel('step'); plt.ylabel('loss'); plt.title(f'Chebyshev-{P} losses | seed={s} | N={N_TRAIN}'); plt.tight_layout(); plt.savefig(AGG/f'seed_{s}_losses.png',dpi=220); plt.show()

    plt.figure(figsize=(7,4))
    for k in LOSS_KEYS: plt.plot(x,[float(r['alpha_'+k]) for r in rs],label=k)
    plt.ylim(-.02,1.02); plt.legend(ncol=2); plt.xlabel('step'); plt.ylabel('alpha'); plt.title(f'Chebyshev-{P} dual weights | seed={s}'); plt.tight_layout(); plt.savefig(AGG/f'seed_{s}_alpha.png',dpi=220); plt.show()

    plt.figure(figsize=(7,4)); plt.plot(x,[float(r['raw_cosine_mean']) for r in rs],label='mean cosine'); plt.plot(x,[float(r['raw_cosine_min']) for r in rs],label='min cosine'); plt.axhline(0,ls='--'); plt.legend(); plt.xlabel('step'); plt.ylabel('cosine'); plt.title(f'Chebyshev-{P} gradient geometry | seed={s}'); plt.tight_layout(); plt.savefig(AGG/f'seed_{s}_geometry.png',dpi=220); plt.show()

    plt.figure(figsize=(7,4)); plt.plot(x,[float(r['min_normalized_alignment']) for r in rs],label='normalized'); plt.plot(x,[float(r['min_raw_alignment']) for r in rs],label='raw'); plt.axhline(0,ls='--'); plt.legend(); plt.xlabel('step'); plt.ylabel('min alignment'); plt.title(f'Chebyshev-{P} alignment | seed={s}'); plt.tight_layout(); plt.savefig(AGG/f'seed_{s}_alignment.png',dpi=220); plt.show()

    plt.figure(figsize=(7,4)); plt.semilogy(x,np.maximum([float(r['fw_gap']) for r in rs],1e-16),label='FW gap'); plt.semilogy(x,np.maximum([float(r['dual_norm']) for r in rs],1e-16),label='dual norm'); plt.legend(); plt.xlabel('step'); plt.title(f'Chebyshev-{P} solver | seed={s}'); plt.tight_layout(); plt.savefig(AGG/f'seed_{s}_solver.png',dpi=220); plt.show()

ks=['rel_l2_u','rel_l2_v','rel_l2_p_aligned']; mu=[mean([float(r[k]) for r in metrics]) for k in ks]; sd=[std([float(r[k]) for r in metrics]) for k in ks]
plt.figure(figsize=(6,4)); xx=np.arange(3); plt.bar(xx,mu,yerr=sd,capsize=5); plt.xticks(xx,['u','v','p aligned']); plt.ylabel('Relative L2'); plt.title(f'Chebyshev-{P} final error | N={N_TRAIN}'); plt.tight_layout(); plt.savefig(AGG/'final_relative_l2_mean_std.png',dpi=220); plt.show()
